# AWS Glue DB Upload

This notebook will be responsible for leveraging AWS Glue to create a table within the raw_data database, which was created within the dependencies folder during setup.

In [1]:
# import libraries
import boto3
import time
from botocore.exceptions import ClientError
from pyathena import connect

In [2]:
# set up glue client and parameters
glue = boto3.client('glue')

crawler_name = 'raw_crawler'
database_name = 'raw_data'
s3_target = 's3://group9-ml-proj-raw-data-bucket-grw/data/raw_merged_data.csv'
glue_role = 'LabRole'

In [3]:
# create crawler for raw data
try:
    response = glue.create_crawler(
        Name=crawler_name,
        Role=glue_role,
        DatabaseName=database_name,
        Targets={'S3Targets': [{'Path': s3_target}]}
    )
    print(f"Crawler '{crawler_name}' created successfully.")
except glue.exceptions.AlreadyExistsException:
    print(f"Crawler ' {crawler_name}' already exists.")

Crawler 'raw_crawler' created successfully.


In [4]:
# run the crawler
glue.start_crawler(Name=crawler_name)

{'ResponseMetadata': {'RequestId': 'b5cb8432-4b26-4538-92ad-c2714153990b',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 24 Mar 2025 23:51:26 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'b5cb8432-4b26-4538-92ad-c2714153990b',
   'cache-control': 'no-cache'},
  'RetryAttempts': 0}}

In [5]:
# update the glue database to skip header lines so that all data can be queried
region_name = 'us-east-1'
s3_staging_dir = "s3://group9-ml-proj-raw-data-bucket-grw/queries/"

In [6]:
# establish athena connection
conn = connect(s3_staging_dir=s3_staging_dir, region_name=region_name)
cursor = conn.cursor()

In [9]:
# run athena query to skip header lines for glue database
query = """
ALTER TABLE raw_data.raw_merged_data_csv
SET TBLPROPERTIES (
  'skip.header.line.count' = '1'
)
"""

In [10]:
# execute alter table statement
cursor.execute(query)

In [11]:
# check the raw_data database
response = glue.get_tables(DatabaseName=database_name)

In [12]:
print(f"Tables in '{database_name}' database:")
for table in response['TableList']:
    print(table['Name'])

Tables in 'raw_data' database:
raw_merged_data_csv


In [1]:
%%html
<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>